# Day 3, session 3A: Trees, ensembles, and XGBoost

**Estimated core time:** 60 minutes  
**Dataset:** `data/customer_churn.csv`  
**Main repetition:** raw data → split → preprocessing pipeline → cross-validation → final test metrics  
**New material:** decision trees, bagging, gradient boosting, and `XGBClassifier`

All exercise cells are intentionally empty. Write the workflow yourself; the repetition is the point.

## Why this matters

On tabular data, interviews and real projects often ask you to compare a simple linear baseline with tree-based models. The algorithm changes, but the trustworthy workflow does not: learned preprocessing stays inside the pipeline, candidates are compared on identical training folds, and the test set is opened only after model selection.

## Learning objectives

By the end, you should be able to:

1. Reconstruct a leakage-safe classification workflow from a raw CSV.
2. Compare logistic regression, a decision tree, random forest, histogram gradient boosting, and XGBoost on the same folds.
3. Explain bagging versus boosting and recognize the main XGBoost controls.
4. Evaluate one selected classifier with accuracy, precision, recall, F1, and ROC AUC.
5. Identify leakage and unfair comparisons in AI-generated model-comparison code.

## R-to-Python bridge

| R / tidymodels idea | Python today |
|---|---|
| recipe + workflow | `ColumnTransformer` inside a `Pipeline` |
| `vfold_cv(..., strata = churned)` | `StratifiedKFold` |
| resampled metric set | `cross_validate(..., scoring=...)` |
| random forest bagging | `RandomForestClassifier` |
| boosted trees | `XGBClassifier` |
| `collect_metrics()` | build a pandas summary from `cross_validate` results |

Tree models do not normally need standardized numeric inputs. We will nevertheless reuse one common preprocessor so model comparisons differ mainly in the estimator, not in data handling.

## The ensemble picture

```text
single tree        one flexible tree; easy to overfit
random forest      many trees independently fit to perturbed data/features, then averaged (bagging)
gradient boosting  trees added sequentially; each new tree corrects current errors
XGBoost            optimized gradient boosting with shrinkage, sampling, and regularization controls
```

High-value XGBoost parameters:

- `n_estimators`: number of boosting rounds / trees
- `learning_rate`: size of each tree's correction
- `max_depth`: interaction complexity of each tree
- `subsample`: fraction of rows available per tree
- `colsample_bytree`: fraction of columns available per tree
- `reg_lambda`: L2 regularization on leaf weights

Smaller `learning_rate` usually needs more trees. Depth and number of trees both raise capacity, but in different ways.

## Suggested pacing

- 0–15 min: rebuild the data split and preprocessor
- 15–25 min: instantiate five model families
- 25–42 min: cross-validate all candidates on identical folds
- 42–52 min: final test evaluation and interpretation
- 52–60 min: leakage audit, retrieval, and interview drill

## Task 1 — Load and audit the raw data

Import the objects you need from pandas, scikit-learn, and XGBoost. Load `data/customer_churn.csv` as `churn`.

Report:

- shape and dtypes,
- missing values by column,
- target prevalence,
- whether `customer_id` is unique.

Before running the cell, predict the shape of `X` after removing the identifier and target.

Useful namespaces/classes include `pandas as pd`, preprocessing and pipeline classes, the five classifiers named below, `train_test_split`, `StratifiedKFold`, `cross_validate`, and classification metrics.

In [50]:
# import libraries for Machine Learning
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    cross_validate,
    StratifiedKFold)
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    precision_score,
    f1_score,
    roc_auc_score
) 
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier
)
from xgboost import XGBClassifier

# import the data
churn = pd.read_csv("data/customer_churn.csv")

In [42]:
print(churn.shape)
print(churn.dtypes)

(2500, 12)
customer_id                str
age                    float64
household_size           int64
tenure_months            int64
contract                   str
internet_service           str
payment_method             str
monthly_charge_eur     float64
monthly_usage_hours    float64
support_calls_6m         int64
late_payments_12m        int64
churned                  int64
dtype: object


In [41]:
# Missing values
churn.isna().sum().sort_values(ascending=False)

monthly_usage_hours    107
age                     83
payment_method          48
customer_id              0
household_size           0
tenure_months            0
contract                 0
internet_service         0
monthly_charge_eur       0
support_calls_6m         0
late_payments_12m        0
churned                  0
dtype: int64

## Task 2 — Rebuild the split and preprocessing pipeline

From memory, create `X`, `y`, `numeric_features`, and `categorical_features`. Exclude `customer_id` and `churned` from `X`.

Reserve 20% as `X_test` / `y_test` with `random_state=42` and `stratify=y`.

Build `preprocessor` with:

- numeric: median imputation, then standardization;
- categorical: most-frequent imputation, then one-hot encoding with unknown categories ignored;
- dense one-hot output via `sparse_output=False`, because all candidate estimators must accept the same transformed matrix.

Do **not** fit the preprocessor yet. Acceptance checks: training and test prevalence are close, and the raw training predictors still contain missing values.

In [43]:
X = churn.drop(columns=["churned", "customer_id"])
y = churn["churned"]
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["str"]).columns

X_train, X_test, y_train, y_test = train_test_split(X,y, stratify=y, test_size=0.2, random_state= 42)

In [33]:
# Build the Preprocessing Pipeline
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

logistic_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

## Task 3 — Trace shapes before fitting models

Fit a **clone** of `preprocessor` to `X_train` only, then transform `X_train` and inspect the result.

Before running it, predict whether the transformed matrix has more or fewer columns than raw `X_train`, and explain why. Confirm:

- raw and transformed shapes,
- transformed object type,
- output feature names with `get_feature_names_out()`.

Also answer: which operations learn values from training data, and which merely apply learned values?

API reminder: `from sklearn.base import clone`.

In [46]:
cloned_preprocessor = clone(preprocessor)
cloned_preprocessor.fit(X_train)
X_train_transformed = cloned_preprocessor.transform(X_train)
X_train_transformed.shape

(2000, 16)

The tranformed matrix has more columsn that tne untransformed X_train. This is because of the OneHotEncoding, that is creating dummy variables for each level within the categroical columns. this is wanted and expected.

In [48]:
cloned_preprocessor.get_feature_names_out()

array(['num__age', 'num__household_size', 'num__tenure_months',
       'num__monthly_charge_eur', 'num__monthly_usage_hours',
       'num__support_calls_6m', 'num__late_payments_12m',
       'cat__contract_monthly', 'cat__contract_one_year',
       'cat__contract_two_year', 'cat__internet_service_dsl',
       'cat__internet_service_fiber', 'cat__internet_service_none',
       'cat__payment_method_bank_transfer', 'cat__payment_method_card',
       'cat__payment_method_electronic_check'], dtype=object)

The imputer learns values from data, the scaler for numeric variabels as well, the encoder, simply applies, nothing learned there. 

## Task 4 — Read the five model specifications

Read these intended settings before writing code:

```python
LogisticRegression(max_iter=1000)
DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=42)
RandomForestClassifier(n_estimators=200, min_samples_leaf=3, n_jobs=-1, random_state=42)
HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15, random_state=42)
XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42,
)
```

For each model, say whether it is linear, one tree, bagged trees, or boosted trees. Predict which models are most sensitive to scaling and which two parameters directly express the learning-rate/number-of-trees trade-off.

Then instantiate the five estimators in a dictionary called `models`.

Answer:

Logistic Regression is Linear.

Decisiion Tree is a single tree classifier.

RandomForestClassifier is a bagged tree.

HistgradientBoost and XGBC are boosted trees. 

In [91]:
models = {
    "logistic_regression" : LogisticRegression(max_iter=1000),
    "decision_tree" : DecisionTreeClassifier(max_depth=4, min_samples_leaf=10, random_state=42),
    "random_forest" : RandomForestClassifier(n_estimators=200, min_samples_leaf=3, n_jobs=-1, random_state=42),
    "hist_gradient_boosting" : HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15, random_state=42),
    "xgboost" : XGBClassifier(
                    n_estimators=200,
                    learning_rate=0.05,
                    max_depth=3,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    reg_lambda=1.0,
                    eval_metric="logloss",
                    n_jobs=-1,
                    random_state=42)
} 


## Task 5 — Wrap every estimator in the same workflow

Create a `Pipeline` for each item in `models`, using a fresh `clone(preprocessor)` followed by that estimator. Store them in a dictionary called `pipelines`.

Acceptance criteria:

- dictionary keys clearly identify the model family;
- every pipeline exposes exactly two outer steps;
- no pipeline has been fitted;
- `customer_id` is never present in the model input.

Why use a clone rather than reusing an already fitted inspection preprocessor?

In [95]:
pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("classifier", clone(model)),
    ])

pipelines.keys()

dict_keys(['logistic_regression', 'decision_tree', 'random_forest', 'hist_gradient_boosting', 'xgboost'])

In [102]:
for pl in pipelines.items():
    print(pl[1])

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['age', 'household_size', 'tenure_months', 'monthly_charge_eur',
       'monthly_usage_hours', 'support_calls_6m', 'late_payments_12m'],
      dtype='str')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                             

## Task 6 — Compare candidates on identical cross-validation folds

Create one `StratifiedKFold` with five folds, shuffling, and `random_state=42`. Define scoring for:

- ROC AUC,
- accuracy,
- precision,
- recall,
- F1.

Loop over `pipelines` and call `cross_validate` using **only** `X_train` and `y_train`. Store fold-level results, then create a tidy summary with one row per model and the mean and standard deviation of each validation metric.

All candidates must receive the same `cv` object and scoring dictionary. Sort by mean validation ROC AUC, descending.

API shape reminder:

```python
results = cross_validate(estimator, X_train, y_train, cv=cv, scoring=scoring)
```

In [105]:
# Step 1: Define the Folds
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

# Step 2: Define the metrics
scoring = {
    "roc_auc" : "roc_auc",
    "precision" : "precision",
    "recall" : "recall",
    "f1" : "f1"
}

results = []

# Step 3: Apply the CV
for name, pipeline in pipelines.items():

    cv_results = cross_validate(
        estimator = pipeline,
        X=X_train,
        y=y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True
        )


    results.append({
        "model": name,
        "roc_auc_mean": cv_results["test_roc_auc"].mean(),
        "roc_auc_sd": cv_results["test_roc_auc"].std(),
        "precision_mean": cv_results["test_precision"].mean(),
        "precision_sd": cv_results["test_precision"].std(),
        "recall_mean": cv_results["test_recall"].mean(),
        "recall_sd": cv_results["test_recall"].std(),
        "f1_mean": cv_results["test_f1"].mean(),
        "f1_sd": cv_results["test_f1"].std()
    })

# Step 4: Convert to DataFrame and sort
clean_df = pd.DataFrame(results)

clean_df = clean_df.sort_values(
    "roc_auc_mean",
    ascending=False
)

clean_df



,model,roc_auc_mean,roc_auc_sd,precision_mean,precision_sd,recall_mean,recall_sd,f1_mean,f1_sd
0,logistic_regression,0.793374,0.015078,0.658874,0.081013,0.360000,0.043350,0.465384,0.055672
4,xgboost,0.774978,0.012771,0.618033,0.051660,0.336842,0.038244,0.434642,0.038382
2,random_forest,0.772611,0.016326,0.650694,0.065800,0.273684,0.021053,0.384606,0.027452
3,hist_gradient_boosting,0.757632,0.014529,0.555534,0.052445,0.347368,0.023062,0.426586,0.027452
1,decision_tree,0.728801,0.010432,0.584757,0.042755,0.273684,0.079611,0.362805,0.071908


## Task 7 — Interpret the cross-validation comparison

Use your summary to answer in plain language:

1. Which candidate has the highest mean ROC AUC?
2. Is its advantage large relative to fold-to-fold variation?
3. Which model has the best recall, and what happens to precision?
4. Does the most complex model clearly justify itself over logistic regression?
5. Why is this comparison fairer than giving each model a different random split?

Store a short written conclusion in a Python string called `cv_conclusion` and display it.

1. logistic regression
2. not not really that much
3. logistic regression has best recall and precision
4. no not at all. log reg seems to outpferorm mor complex model, whilst being simpler 
5.  because it cant happen that 1 model gets a random. "easy" split to train on, they all get the same, and therefore we can compare performances more clearer.

## Task 8 — Fit once and evaluate once on the untouched test set

Select one workflow using the cross-validation results—not test performance. Fit that complete pipeline to all of `X_train` and `y_train`.

On `X_test`, calculate:

- accuracy,
- precision,
- recall,
- F1,
- ROC AUC from positive-class probabilities.

Put the results in a one-row `DataFrame`. For every metric, state whether larger or smaller is better and interpret at least two values in ordinary language.

Acceptance criteria: test data appears only in `predict`, `predict_proba`, and metric calls after model selection.

In [120]:
# Slecect xgbost
# Fit the model
xgb_fit = pipelines["xgboost"].fit(X_train,y_train)
# Get predictions 
predictions = xgb_fit.predict(X_test)
predictions_proba = xgb_fit.predict_proba(X_test)[:,1]


# Compute Scores
accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, zero_division=0)
recall = recall_score(y_test, predictions)
f1 = f1_score(y_test, predictions)
roc_auc = roc_auc_score(y_test, predictions_proba)

print(accuracy)
print(precision)
print(recall)
print(f1)
print(roc_auc)

0.82
0.7230769230769231
0.3949579831932773
0.5108695652173914
0.8195152076578663


## Task 9 — Debug an unfair and leaky comparison

An AI assistant proposes:

```python
encoded = preprocessor.fit_transform(X)

logistic_score = cross_val_score(
    LogisticRegression(), encoded, y, cv=5, scoring="roc_auc"
).mean()

xgb_score = cross_val_score(
    XGBClassifier(), encoded, y, cv=10, scoring="accuracy"
).mean()

if xgb_score > logistic_score:
    final_model = XGBClassifier().fit(preprocessor.transform(X_test), y_test)
```

List at least five problems. Then rewrite a safe comparison using the already reserved training data, one shared splitter, one shared metric set, and full pipelines.

Hint: inspect preprocessing, resampling, metrics, test-set role, and whether the final estimator matches the evaluated workflow.

## Retrieval practice — Task 10: navigate the fitted pipeline

Without looking back at Day 2, retrieve from your fitted final workflow:

1. outer step names,
2. fitted preprocessor,
3. fitted categorical pipeline,
4. learned one-hot categories,
5. fitted estimator and two of its important hyperparameters,
6. transformed test-matrix shape.

Then explain the difference between `named_steps` and `named_transformers_`, and between `transform` and `predict`.

## Interview questions

Answer aloud or add markdown cells:

1. What do bagging and boosting change, and how does a random forest differ from gradient boosting?
2. Why can XGBoost be excellent on tabular data, and when might logistic regression still be preferable?
3. How do `learning_rate` and `n_estimators` interact?
4. Why must preprocessing remain inside every cross-validated pipeline even for tree models?

## Self-check and reflection

Record:

- Confidence from 1–5:
- Could you rebuild the pipeline faster than on Day 2?
- Can you explain random forest versus gradient boosting without code?
- Which XGBoost control is least clear?
- One thing you can now explain in an interview:
- What deserves another repetition?

## Optional extension — Early stopping (outside the 60 minutes)

Research the current `XGBClassifier.fit` API in the installed version and fit an XGBoost model with a validation set and early stopping. Keep the untouched test set out of the stopping decision.

Explain why this is less convenient inside ordinary cross-validation pipelines: early stopping needs a fold-specific validation set that has undergone the same fitted preprocessing. Compare the chosen number of boosting rounds with the fixed 200-round model.

## Finish line

You have completed the core lesson when you can explain and reproduce:

**raw classification data → held-out test set → common preprocessing → identical CV folds → model-family comparison → one selected workflow → one final test evaluation**